In [5]:
import pandas as pd
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    csv_path = Path(next(iter(uploaded.keys())))
else:
    csv_path = Path("../data/raw/gpu_specs.csv")

df = pd.read_csv(csv_path)
print("rows, cols:", df.shape)

rows, cols: (3203, 134)


In [6]:
print("n_columns:", df.shape[1])
print(df.dtypes.value_counts())

n_columns: 134
object     116
float64     16
int64        2
Name: count, dtype: int64


In [13]:
import numpy as np

n = len(df)

# Numeric: missing → n_nan / nan_pct (+ n_inf, p99, max).
# Text/object: missing → n_null only (pandas NaN count). Use n_unique_non_nan for cardinality.

def profile_numeric(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in frame.columns:
        s = frame[c]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        nn = int(s.isna().sum())
        sn = s.dropna().to_numpy(dtype=float, copy=False)
        rec = {
            "column": c,
            "dtype": str(s.dtype),
            "n_nan": nn,
            "nan_pct": round(100 * nn / n, 2),
            "n_inf": int(np.isinf(sn).sum()) if sn.size else 0,
        }
        fin = sn[np.isfinite(sn)]
        if fin.size:
            rec["p99"] = float(np.percentile(fin, 99))
            rec["max"] = float(fin.max())
        else:
            rec["p99"] = float("nan")
            rec["max"] = float("nan")
        rows.append(rec)
    return pd.DataFrame(rows).sort_values("nan_pct", ascending=False)


def profile_text(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in frame.columns:
        s = frame[c]
        if pd.api.types.is_numeric_dtype(s):
            continue
        rows.append(
            {
                "column": c,
                "dtype": str(s.dtype),
                "n_null": int(s.isna().sum()),
                "n_unique_non_nan": int(s.nunique(dropna=True)),
            }
        )
    return pd.DataFrame(rows).sort_values(["n_null", "column"], ascending=[False, True])


prof_num = profile_numeric(df)
prof_obj = profile_text(df)

print("numeric columns:", len(prof_num), "| fields: n_nan, nan_pct, n_inf, p99, max")
display(prof_num)

print("text / object columns:", len(prof_obj), "| fields: n_null, n_unique_non_nan")
display(prof_obj)

numeric columns: 18 | fields: n_nan, nan_pct, n_inf, p99, max


,column,dtype,n_nan,nan_pct,n_inf,p99,max
7,Render Config__MCM,float64,3192,99.66,0,2.00,2.0
14,Graphics Features__CUDA SDK,float64,3191,99.63,0,6.50,6.5
11,Render Config__XMX Cores,float64,3172,99.03,0,1024.00,1024.0
8,Render Config__Matrix Cores,float64,3171,99.00,0,1216.00,1216.0
10,Graphics Processor__Generation,float64,3167,98.88,0,13.00,13.0
16,Render Config__SMM Count,float64,3104,96.91,0,24.00,24.0
15,Render Config__SMX Count,float64,3043,95.00,0,15.00,15.0
9,Render Config__Execution Units,float64,3014,94.10,0,911.36,1024.0
17,Render Config__Tensor Cores,float64,2916,91.04,0,752.00,752.0
6,Render Config__RT Cores,float64,2840,88.67,0,188.00,188.0


text / object columns: 116 | fields: n_null, n_unique_non_nan


,column,dtype,n_null,n_unique_non_nan
87,Clock Speeds__Peak Clock,object,3202,1
89,Clock Speeds__UVD Clock,object,3202,1
114,Graphics Features__NVDEC,object,3202,1
113,Graphics Features__NVENC,object,3202,1
88,Board Design__BIOS Number,object,3201,2
...,...,...,...,...
9,Top__BUS WIDTH,object,0,39
2,Top__GRAPHICS PROCESSOR,object,0,603
7,Top__MEMORY SIZE,object,0,80
8,Top__MEMORY TYPE,object,0,33


In [ ]:
# Cleanliness: columns with *any* missing values (judge keep / drop / impute from here)

n_rows = len(df)

print("=== numeric columns — pandas NaN (n_nan) ===")
num_any = prof_num[prof_num["n_nan"] > 0].sort_values("n_nan", ascending=False)
print("count with ≥1 NaN:", len(num_any), "of", len(prof_num), "numeric")
print("count all-NaN (useless column):", int((prof_num["n_nan"] == n_rows).sum()))
print("count with inf:", int((prof_num["n_inf"] > 0).sum()))
display(num_any)

extreme = prof_num[
    prof_num["p99"].notna() & prof_num["max"].notna() & (prof_num["max"] > 100 * prof_num["p99"])
]
print("numeric: max >> p99 (spot outliers), count:", len(extreme))
display(extreme[["column", "p99", "max"]] if len(extreme) else extreme)

print("\n=== text / object columns — pandas null (n_null) ===")
obj_any = prof_obj[prof_obj["n_null"] > 0].sort_values("n_null", ascending=False)
print("count with ≥1 null:", len(obj_any), "of", len(prof_obj), "object")
print("count all-null (empty column):", int((prof_obj["n_null"] == n_rows).sum()))
display(obj_any)

print("\n--- names only (copy-paste) ---")
print("numeric_with_any_nan =", num_any["column"].tolist())
print("object_with_any_null =", obj_any["column"].tolist())

=== numeric columns — pandas NaN (n_nan) ===
count with ≥1 NaN: 16 of 18 numeric
count all-NaN (useless column): 0
count with inf: 0


,column,dtype,n_nan,nan_pct,n_inf,p99,max
7,Render Config__MCM,float64,3192,99.66,0,2.00,2.0
14,Graphics Features__CUDA SDK,float64,3191,99.63,0,6.50,6.5
11,Render Config__XMX Cores,float64,3172,99.03,0,1024.00,1024.0
8,Render Config__Matrix Cores,float64,3171,99.00,0,1216.00,1216.0
10,Graphics Processor__Generation,float64,3167,98.88,0,13.00,13.0
16,Render Config__SMM Count,float64,3104,96.91,0,24.00,24.0
15,Render Config__SMX Count,float64,3043,95.00,0,15.00,15.0
9,Render Config__Execution Units,float64,3014,94.10,0,911.36,1024.0
17,Render Config__Tensor Cores,float64,2916,91.04,0,752.00,752.0
6,Render Config__RT Cores,float64,2840,88.67,0,188.00,188.0


numeric: max >> p99 (spot outliers), count: 0


,column,dtype,n_nan,nan_pct,n_inf,p99,max



=== text / object columns — pandas null (n_null) ===
count with ≥1 null: 98 of 116 object
count all-null (empty column): 0


,column,dtype,n_null,n_unique_non_nan
87,Clock Speeds__Peak Clock,object,3202,1
89,Clock Speeds__UVD Clock,object,3202,1
114,Graphics Features__NVDEC,object,3202,1
113,Graphics Features__NVENC,object,3202,1
88,Board Design__BIOS Number,object,3201,2
...,...,...,...,...
5,Top__TMUS,object,29,111
11,Graphics Processor__Architecture,object,25,94
6,Top__ROPS,object,25,45
31,Board Design__Outputs,object,7,165



--- names only (copy-paste) ---
numeric_with_any_nan = ['Render Config__MCM', 'Graphics Features__CUDA SDK', 'Render Config__XMX Cores', 'Render Config__Matrix Cores', 'Graphics Processor__Generation', 'Render Config__SMM Count', 'Render Config__SMX Count', 'Render Config__Execution Units', 'Render Config__Tensor Cores', 'Render Config__RT Cores', 'Render Config__Vertex Shaders', 'Render Config__Pixel Shaders', 'Render Config__SM Count', 'Render Config__Compute Units', 'Graphics Features__CUDA', 'Render Config__Shading Units']
object_with_any_null = ['Clock Speeds__Peak Clock', 'Clock Speeds__UVD Clock', 'Graphics Features__NVDEC', 'Graphics Features__NVENC', 'Board Design__BIOS Number', 'Board Design__Base Storage', 'Clock Speeds__SOC Clock', 'Graphics Processor__S-Spec', 'Integrated Graphics__Launch Price', 'Mobile Graphics__Announced', 'Graphics Processor__Package Size', 'Graphics Processor__MCD Size', 'Graphics Processor__GCD Size', 'Board Design__Inputs', 'Theoretical Performance